In [131]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,image_path,label
0,train/img_000001.jpg,1
1,train/img_000002.jpg,1
2,train/img_000003.jpg,1
3,train/img_000004.jpg,0
4,train/img_000005.jpg,1


In [141]:
import torch

epochs = 11
learning_rate = 0.002
batch_size= 32

device = torch.device("cuda")
device

device(type='cuda')

In [142]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2

class MyDataset(Dataset):
    def __init__(self, df, transform, img_size=128):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        label = torch.tensor(row.label, dtype=torch.float32).view(1)
        return img, label

train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor()
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.ToTensor()
])

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, random_state=42, test_size=0.2)

train_dataset = MyDataset(df_1, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

test_dataset = MyDataset(df_2, test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [143]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        x = self.net(x)
        return x

model = CNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

CNN(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): LazyLinear(in_features=0, out_features=64, bias=True)
    (11): ReLU()
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): ReLU()
    (14): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [144]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Training Loss: {total_loss / len(train_loader):.4f}")


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader):.4f}')

In [145]:
for epoch in range(1, epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, test_loader, criterion)

Epoch 1, Training Loss: 0.6593
Test Loss: 0.6154
Epoch 2, Training Loss: 0.5482
Test Loss: 0.3961
Epoch 3, Training Loss: 0.4358
Test Loss: 0.3058
Epoch 4, Training Loss: 0.3586
Test Loss: 0.2494
Epoch 5, Training Loss: 0.2979
Test Loss: 0.2395
Epoch 6, Training Loss: 0.2607
Test Loss: 0.2436
Epoch 7, Training Loss: 0.2371
Test Loss: 0.2021
Epoch 8, Training Loss: 0.1938
Test Loss: 0.2139
Epoch 9, Training Loss: 0.2118
Test Loss: 0.1920
Epoch 10, Training Loss: 0.2256
Test Loss: 0.2391
Epoch 11, Training Loss: 0.1826
Test Loss: 0.2114


In [146]:
class FinalDataset(Dataset):
    def __init__(self, df, transform, img_size=128):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        return img

final_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.ToTensor()
])

final_dataset = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

In [147]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        results = torch.sigmoid(model(inputs)).cpu()
        preds.append(results)

preds = torch.cat(preds, dim=0).numpy().flatten()
preds = (preds >= 0.5).astype(int)
preds[:10]

array([1, 0, 1, 1, 1, 0, 1, 1, 0, 1])

In [148]:
df_preds = pd.DataFrame({
    "image_path": df_test.image_path,
    "label": preds
})

df_preds

,image_path,label
0,test/img_000615.jpg,1
1,test/img_000616.jpg,0
2,test/img_000617.jpg,1
3,test/img_000618.jpg,1
4,test/img_000619.jpg,1
...,...,...
611,test/img_001226.jpg,1
612,test/img_001227.jpg,1
613,test/img_001228.jpg,0
614,test/img_001229.jpg,0


In [149]:
df_preds.to_csv("submission.csv", index=False)